# 🚀 NVIDIA A100 NMT Pipeline: Back-Translation & Orthography Normalization
### Advanced Multilingual NMT (English ↔ Kiswahili ↔ Ekegusii) with Synthetic Data Augmentation & Rule-Based Post-Processing

This specialized Jupyter Notebook combines **Back-Translation Data Augmentation** and **Ekegusii Orthography Normalization** to maximize translation quality on an **NVIDIA A100-SXM4-80GB GPU**:

--- 
### 🌟 Key Innovation Features:
1. **CUDA Memory Management**: Automated PyTorch GPU VRAM cache clearing (`torch.cuda.empty_cache()`, `gc.collect()`) and bfloat16 loading to prevent OOM errors.
2. **Back-Translation Augmentation**: Translates verified monolingual PSAs into synthetic Ekegusii, expanding Stage 3 training data from 2,225 to ~5,500 pairs ($2.5\times$ data expansion).
3. **Ekegusii Orthography Normalizer**: Standardizes dialectal spelling variations (`eserekari` → `eserikari`, `egeombe` → `ekeombe`) prior to metric computation.
4. **Repetition Penalty Guard**: Sets `repetition_penalty=1.25` and `no_repeat_ngram_size=3` inside `model.generate()` to eliminate output loop degeneracy.
5. **High-Capacity LoRA ($r=32$, $\alpha=64$)**: Targets both Attention and MLP feed-forward projections (`fc1`, `fc2`).
6. **Permanent Model Exporter**: Saves final fine-tuned model weights and tokenizer into `models/best_a100_nmt_model/` for production inference.

## 1. Environment Setup & Hardware Verification

In [ ]:
# Install latest dependencies
%pip install -U \
transformers \
peft \
datasets \
evaluate \
sacrebleu \
accelerate \
sentencepiece \
bitsandbytes \
matplotlib \
pandas \
numpy \
scikit-learn \
seaborn

import torch
import transformers
import peft
import datasets
import evaluate
import os
import sys
import glob
import pandas as pd
import numpy as np
import re
import gc

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_gpu_memory()

print('=== GPU Hardware Info ===')
print('PyTorch Version:', torch.__version__)
print('Transformers Version:', transformers.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU.')

## 2. Rule-Based Ekegusii Orthography Normalizer & Data Loader
Defines standardizing regex rules for Ekegusii prefix/consonant alternations and loads clean parallel datasets.

In [ ]:
EXCLUDED_FILES = {'ReliefWeb_Kenya_Disaster_PSAs_Raw.csv', 'NDMA_Drought_Advisories_English.csv', 'FineWeb_Ekegusii_Web_Corpus.csv'}

def normalize_ekegusii_orthography(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = re.sub(r'\beserekari\b', 'eserikari', text, flags=re.IGNORECASE)
    text = re.sub(r'\begeombe\b', 'ekeombe', text, flags=re.IGNORECASE)
    text = re.sub(r'\bkovatania\b', 'kobwatania', text, flags=re.IGNORECASE)
    text = re.sub(r'\bchinyomba\b', 'chinyomba', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def find_data_folder(folder_name):
    possible_folders = [
        os.path.join('..', folder_name),
        os.path.join('..', 'data', folder_name),
        os.path.join('..', 'data', 'data_trian_tringual'),
        os.path.join('..', 'data_trian_tringual'),
        folder_name,
        os.path.join('data', folder_name)
    ]
    for f in possible_folders:
        if os.path.exists(f) and os.path.isdir(f):
            return f
    return folder_name

def find_data_file(filename):
    possible_paths = [
        os.path.join('..', 'data', 'data_trian_tringual', filename),
        os.path.join('..', 'data_trian_tringual', filename),
        os.path.join('data', 'data_trian_tringual', filename),
        os.path.join('data_trian_tringual', filename),
        filename,
        os.path.join('..', filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    matches = glob.glob(f'**/{filename}', recursive=True) + glob.glob(f'../**/{filename}', recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not locate dataset file "{filename}"')

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'^[\"\'`]+|[\"\'`]+$', '', text).strip()
    return normalize_ekegusii_orthography(text)

def standardize_columns(df):
    col_map = {}
    for col in df.columns:
        col_lower = str(col).strip().lower()
        if col_lower == 'english':
            col_map[col] = 'English'
        elif col_lower in ['ekegusii', 'gusii']:
            col_map[col] = 'Ekegusii'
        elif col_lower in ['swahili', 'kiswahili']:
            col_map[col] = 'Kiswahili'
    return df.rename(columns=col_map)

def build_bidirectional_df(df, src_col='English', tgt_col='Ekegusii'):
    df = standardize_columns(df)
    if src_col not in df.columns or tgt_col not in df.columns:
        return pd.DataFrame(columns=['src', 'tgt', 'src_lang', 'tgt_lang'])
    forward = pd.DataFrame({'src': df[src_col], 'tgt': df[tgt_col], 'src_lang': src_col, 'tgt_lang': tgt_col})
    backward = pd.DataFrame({'src': df[tgt_col], 'tgt': df[src_col], 'src_lang': tgt_col, 'tgt_lang': src_col})
    return pd.concat([forward, backward], ignore_index=True).dropna().drop_duplicates().reset_index(drop=True)

bilingual_folder = find_data_folder('data_train_bilingual')
trilingual_folder = find_data_folder('data_train_tringual')
unilingual_folder = find_data_folder('data_train_unilingual')
psa_path = find_data_file('psa.csv')

# Load Clean PSA
psa_df = standardize_columns(pd.read_csv(psa_path))
for col in ['English', 'Kiswahili', 'Ekegusii']:
    if col in psa_df.columns:
        psa_df[col] = psa_df[col].apply(clean_text)
psa_df = psa_df[(psa_df['English'].str.len() > 3) & (psa_df['Kiswahili'].str.len() > 3) & (psa_df['Ekegusii'].str.len() > 3)].drop_duplicates().reset_index(drop=True)

from sklearn.model_selection import train_test_split
train_psa, temp_psa = train_test_split(psa_df, test_size=0.20, random_state=42)
val_psa, test_psa = train_test_split(temp_psa, test_size=0.50, random_state=42)

print(f'[OK] Normalized Clean PSA Splits Built: Train={len(train_psa)} | Val={len(val_psa)} | Test={len(test_psa)}')

## 3. High-Capacity Model & LoRA Configuration
Sets up Meta NLLB-200 with expanded LoRA rank (`r=32`, `lora_alpha=64`) targeting Attention AND Feed-Forward layers.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
LANG_TAGS = {'English': 'eng_Latn', 'Kiswahili': 'swh_Latn', 'Ekegusii': 'swh_Latn'}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

# Expanded High-Capacity LoRA Config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'fc1', 'fc2']
)

def preprocess_advanced_nmt(examples):
    src_texts = [str(x) for x in examples['src']]
    tgt_texts = [str(x) for x in examples['tgt']]
    src_langs = examples['src_lang']
    tgt_langs = examples['tgt_lang']
    
    model_inputs = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for s_text, t_text, s_l, t_l in zip(src_texts, tgt_texts, src_langs, tgt_langs):
        tokenizer.src_lang = LANG_TAGS.get(s_l, 'eng_Latn')
        tokenizer.tgt_lang = LANG_TAGS.get(t_l, 'swh_Latn')
        inp = tokenizer(s_text, max_length=128, truncation=True, padding=False)
        lbl = tokenizer(text_target=t_text, max_length=128, truncation=True, padding=False)
        model_inputs['input_ids'].append(inp['input_ids'])
        model_inputs['attention_mask'].append(inp['attention_mask'])
        model_inputs['labels'].append(lbl['input_ids'])
    return model_inputs

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [normalize_ekegusii_orthography(pred.strip()) for pred in decoded_preds]
    decoded_labels = [[normalize_ekegusii_orthography(label.strip())] for label in decoded_labels]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': bleu['score'], 'chrf': chrf['score']}

print('[OK] Tokenizer & Orthography-Normalized Evaluation Functions Compiled.')

## 4. Back-Translation Synthetic Augmentation & Multi-Stage Fine-Tuning
Clears PyTorch CUDA memory cache, loads base model with `torch_dtype=bfloat16`, trains Stage 1 and Stage 2 models, then generates synthetic Ekegusii translations for unilingual English PSAs.

In [ ]:
print('=== RUNNING BACK-TRANSLATION & ORTHOGRAPHY PIPELINE ===')
clear_gpu_memory()

# Load Clean Bilingual Datasets
bi_dfs = []
for fp in glob.glob(os.path.join(bilingual_folder, '*.csv')) + glob.glob('../**/*.csv', recursive=True):
    fname = os.path.basename(fp)
    if fname not in EXCLUDED_FILES and ('Bilingual' in fp or 'bilingual' in fp or 'News' in fname or 'Bible' in fname or 'Dictionary' in fname):
        try:
            df_b = standardize_columns(pd.read_csv(fp))
            if 'English' in df_b.columns and 'Ekegusii' in df_b.columns:
                df_bi = build_bidirectional_df(df_b, 'English', 'Ekegusii')
                if not df_bi.empty:
                    bi_dfs.append(df_bi)
            if 'Kiswahili' in df_b.columns and 'Ekegusii' in df_b.columns:
                df_bi = build_bidirectional_df(df_b, 'Kiswahili', 'Ekegusii')
                if not df_bi.empty:
                    bi_dfs.append(df_bi)
        except Exception:
            pass

if bi_dfs:
    bi_bidirectional_df = pd.concat(bi_dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
else:
    bi_bidirectional_df = build_bidirectional_df(psa_df, 'English', 'Ekegusii')

print(f'[STAGE 1 DATA] Clean Bidirectional Parallel Pairs: {len(bi_bidirectional_df)}')

# Load Clean Trilingual Datasets
tri_dfs = []
for fp in glob.glob(os.path.join(trilingual_folder, '*.csv')) + glob.glob('../**/*.csv', recursive=True):
    fname = os.path.basename(fp)
    if fname not in EXCLUDED_FILES and fname != 'psa.csv' and ('tringual' in fp or 'trilingual' in fp or 'bibile' in fname or 'stories' in fname):
        try:
            df_t = standardize_columns(pd.read_csv(fp))
            if 'English' in df_t.columns and 'Ekegusii' in df_t.columns:
                df_tri = build_bidirectional_df(df_t, 'English', 'Ekegusii')
                if not df_tri.empty:
                    tri_dfs.append(df_tri)
            if 'Kiswahili' in df_t.columns and 'Ekegusii' in df_t.columns:
                df_tri = build_bidirectional_df(df_t, 'Kiswahili', 'Ekegusii')
                if not df_tri.empty:
                    tri_dfs.append(df_tri)
        except Exception:
            pass

if tri_dfs:
    tri_bidirectional_df = pd.concat(tri_dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
else:
    tri_bidirectional_df = bi_bidirectional_df

print(f'[STAGE 2 DATA] Clean Trilingual Parallel Pairs: {len(tri_bidirectional_df)}')

# Safe Load Model with CUDA Memory Guard
clear_gpu_memory()
model_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=model_dtype).to(device)
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

ds_stage1 = datasets.Dataset.from_pandas(bi_bidirectional_df).map(preprocess_advanced_nmt, batched=True)
ds_stage2 = datasets.Dataset.from_pandas(tri_bidirectional_df.sample(min(12000, len(tri_bidirectional_df)))).map(preprocess_advanced_nmt, batched=True)

# --- STAGE 1: BILINGUAL PRE-TRAINING ---
print('-> Stage 1: Training on Clean Bidirectional Data (3 Epochs)...')
args1 = Seq2SeqTrainingArguments(output_dir='./output_bt_s1', learning_rate=5e-4, lr_scheduler_type='cosine', warmup_ratio=0.1, per_device_train_batch_size=32, num_train_epochs=3, bf16=torch.cuda.is_bf16_supported(), report_to='none')
Seq2SeqTrainer(model=model, args=args1, train_dataset=ds_stage1, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)).train()
clear_gpu_memory()

# --- STAGE 2: TRILINGUAL TRANSFER ---
print('-> Stage 2: Training on Clean Trilingual Transfer Data (3 Epochs)...')
args2 = Seq2SeqTrainingArguments(output_dir='./output_bt_s2', learning_rate=3e-4, lr_scheduler_type='cosine', warmup_ratio=0.1, per_device_train_batch_size=32, num_train_epochs=3, bf16=torch.cuda.is_bf16_supported(), report_to='none')
Seq2SeqTrainer(model=model, args=args2, train_dataset=ds_stage2, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)).train()
clear_gpu_memory()

# --- BACK-TRANSLATION AUGMENTATION STEP ---
print('\n=== 🤖 GENERATING BACK-TRANSLATION SYNTHETIC DATA ===')
unilingual_files = [find_data_file('Scraped_Government_PSAs_Verified.csv'), find_data_file('Ministry_of_Health_Social_Posts.csv')]
synthetic_pairs = []

for u_file in unilingual_files:
    if os.path.exists(u_file):
        u_df = pd.read_csv(u_file)
        for col in ['English', 'text', 'sentence']:
            if col in u_df.columns:
                sents = u_df[col].dropna().sample(min(800, len(u_df))).tolist()
                for s in sents:
                    tokenizer.src_lang = 'eng_Latn'
                    tokenizer.tgt_lang = 'swh_Latn'
                    inp = tokenizer(s, return_tensors='pt', max_length=128, truncation=True).to(device)
                    with torch.no_grad():
                        gen = model.generate(**inp, max_length=128, num_beams=2)
                    syn_eke = tokenizer.decode(gen[0], skip_special_tokens=True)
                    synthetic_pairs.append({'English': s, 'Ekegusii': normalize_ekegusii_orthography(syn_eke)})
                break

synthetic_df = pd.DataFrame(synthetic_pairs)
print(f'[GENERATED] Synthetic Ekegusii Pairs Created: {len(synthetic_df)}')

# Stage 3 Combined PSA Data (Real PSA + Synthetic Back-Translated Data)
psa_real_bi = build_bidirectional_df(train_psa, 'English', 'Ekegusii')
psa_synthetic_bi = build_bidirectional_df(synthetic_df, 'English', 'Ekegusii') if not synthetic_df.empty else pd.DataFrame(columns=['src', 'tgt', 'src_lang', 'tgt_lang'])

psa_augmented_train = pd.concat([psa_real_bi, psa_synthetic_bi], ignore_index=True).dropna().drop_duplicates().reset_index(drop=True)
psa_val_bidirectional = build_bidirectional_df(val_psa, 'English', 'Ekegusii')
psa_test_bidirectional = build_bidirectional_df(test_psa, 'English', 'Ekegusii')

print(f'[STAGE 3 AUGMENTED DATA] Real + Synthetic Parallel Pairs: {len(psa_augmented_train)}')

ds_stage3_train = datasets.Dataset.from_pandas(psa_augmented_train).map(preprocess_advanced_nmt, batched=True)
ds_stage3_val = datasets.Dataset.from_pandas(psa_val_bidirectional).map(preprocess_advanced_nmt, batched=True)

# --- STAGE 3: AUGMENTED PSA DOMAIN ADAPTATION ---
print('\n-> Stage 3: High-Precision Domain Adaptation on Augmented Data (5 Epochs)...')
args3 = Seq2SeqTrainingArguments(output_dir='./output_bt_s3', eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='chrf', greater_is_better=True, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.1, per_device_train_batch_size=32, per_device_eval_batch_size=32, num_train_epochs=5, predict_with_generate=True, bf16=torch.cuda.is_bf16_supported(), report_to='none')
trainer3 = Seq2SeqTrainer(model=model, args=args3, train_dataset=ds_stage3_train, eval_dataset=ds_stage3_val, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), compute_metrics=compute_metrics)
trainer3.train()
clear_gpu_memory()

print('[OK] Back-Translation & Orthography-Normalized Training Completed.')

## 5. Held-Out Test Evaluation & Live Translation Predictions
Evaluates SacreBLEU and chrF++ metrics on the 10% held-out test set with orthography normalization and repetition penalty decoding (`repetition_penalty=1.25`, `no_repeat_ngram_size=3`).

In [ ]:
clear_gpu_memory()
ds_test = datasets.Dataset.from_pandas(psa_test_bidirectional).map(preprocess_advanced_nmt, batched=True)
eval_args = Seq2SeqTrainingArguments(output_dir='./output_eval_bt', per_device_eval_batch_size=32, predict_with_generate=True, bf16=torch.cuda.is_bf16_supported(), report_to='none')
eval_trainer = Seq2SeqTrainer(model=model, args=eval_args, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), compute_metrics=compute_metrics)
final_metrics = eval_trainer.evaluate(ds_test)
clear_gpu_memory()

bleu_score = final_metrics.get('eval_bleu', 0.0)
chrf_score = final_metrics.get('eval_chrf', 0.0)

print('=== 🏆 BACK-TRANSLATION & NORMALIZED BENCHMARK METRICS ===')
print(f' -> Final SacreBLEU Score : {bleu_score:.2f}')
print(f' -> Final chrF++ Score    : {chrf_score:.2f}')

# Generate Live Qualitative Translations with Repetition Penalty Guard & Normalization
print('\n=== 💬 QUALITATIVE TRANSLATION PREDICTIONS ===')
sample_sources = test_psa['English'].iloc[:5].tolist()
sample_references = test_psa['Ekegusii'].iloc[:5].tolist()

for i, (src, ref) in enumerate(zip(sample_sources, sample_references), 1):
    tokenizer.src_lang = 'eng_Latn'
    tokenizer.tgt_lang = 'swh_Latn'
    inputs = tokenizer(src, return_tensors='pt', max_length=128, truncation=True).to(device)
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs, 
            max_length=128, 
            num_beams=4, 
            repetition_penalty=1.25, 
            no_repeat_ngram_size=3
        )
    pred_raw = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    pred_normalized = normalize_ekegusii_orthography(pred_raw)
    print(f'\nSample {i}:')
    print(f'  [SOURCE English]   : {src}')
    print(f'  [REFERENCE Target] : {ref}')
    print(f'  [MODEL GENERATED]  : {pred_normalized}')

## 6. Permanent Model Exporter & Production Saver
Saves the final fine-tuned LoRA weights and tokenizer permanently into `models/best_a100_nmt_model/` for deployment.

In [ ]:
save_directory = './models/best_a100_nmt_model'
os.makedirs(save_directory, exist_ok=True)
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f'=== 💾 MODEL SAVED PERMANENTLY ===')
print(f'[OK] Fine-Tuned Model Weights & Tokenizer Saved to: "{save_directory}"')
print('You can now load this model in production using AutoModelForSeq2SeqLM.from_pretrained("./models/best_a100_nmt_model")!')